# 03 — Build the First Standardized Dataset

This notebook prepares the conventional laser-welding records for later modelling. It does **not** combine different studies, delete zero measurements, average repeats, or overwrite raw data.

**Input:** `data/raw/Laser Welding.xlsx`  
**Output:** `data/processed/conventional_laser_welding_standardized.csv`


In [ ]:
from pathlib import Path
import pandas as pd

RAW_FILE = Path('data/raw/Laser Welding.xlsx')
OUT_DIR = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError(f'Cannot find {RAW_FILE}. Check that the repository files are available in this Colab session.')

raw = pd.read_excel(RAW_FILE, sheet_name='Data')
print('Rows:', len(raw))
print('Original columns:', raw.columns.tolist())
display(raw.head())


## 1. Map the source columns

We retain the source measurements and assign consistent names for analysis. If a source heading changes, this cell will stop rather than silently guessing.

In [ ]:
column_map = {
    'Laser Power [W]': 'laser_power_W',
    'Welding Speed [mm/s]': 'welding_speed_mm_s',
    'Weld bead width [mm]': 'weld_bead_width_mm',
    'Interface width [mm]': 'interface_width_mm',
    'Weld bead penetration depth [mm]': 'penetration_depth_mm',
}

missing_columns = [c for c in column_map if c not in raw.columns]
if missing_columns:
    raise ValueError(f'Expected source columns not found: {missing_columns}')

df = raw[list(column_map)].rename(columns=column_map).copy()
df.insert(0, 'source_id', 'conventional_laser_welding_dataset')
df.insert(1, 'study_id', 'source_dataset_01')
df.insert(2, 'record_id', [f'CLW_{i+1:03d}' for i in range(len(df))])

# Replicate number within each identical process-parameter combination.
df['replicate_id'] = (
    df.groupby(['laser_power_W', 'welding_speed_mm_s']).cumcount() + 1
)
df['weld_observation_zero_flag'] = (
    df[['weld_bead_width_mm', 'interface_width_mm', 'penetration_depth_mm']]
      .eq(0).all(axis=1)
)
df['data_status'] = 'transcribed_from_source_dataset; zero status not yet verified'

display(df.head(10))


## 2. Run quality checks

These checks report issues; they do not automatically modify the measurements.

In [ ]:
print('Shape:', df.shape)
print('Missing values by column:')
display(df.isna().sum().rename('missing_count').to_frame())

print('Zero-output records:', int(df['weld_observation_zero_flag'].sum()))
print('Records by process condition:')
display(df.groupby(['laser_power_W', 'welding_speed_mm_s']).size().rename('replicate_count').reset_index())

print('Duplicate complete measurement rows:', int(df.duplicated().sum()))
print('No rows have been removed or averaged.')


## 3. Save the standardized dataset

The raw Excel file remains untouched. This CSV is a derived working copy with traceability columns.

In [ ]:
output_file = OUT_DIR / 'conventional_laser_welding_standardized.csv'
df.to_csv(output_file, index=False)
print('Saved:', output_file)
print('Rows saved:', len(pd.read_csv(output_file)))
print('Raw source unchanged:', RAW_FILE.exists())


## 4. Download the output from Colab

The file is saved in the current Colab runtime, not automatically in GitHub. To download it, run:

In [ ]:
from google.colab import files
files.download(str(output_file))
